# UKRI FoR Classifier — One-off Local Backfill

This notebook is intentionally separate from the weekly S3 DELTA notebook.

It reuses the Version 2 inference approach but is simplified for the initial historical backfill:

- Reads the historical Parquet files from `PROJECT_ROOT / "prod_data_temp"`.
- Does **not** use S3 input discovery or filename timestamp rules.
- Processes all local Parquet files sequentially to avoid loading the whole backfill into memory.
- Deduplicates globally using `ApplicationID + ApplicationOriginSource`.
- Uses only `ApplicationTitle + ApplicationSummary` for inference.
- Rows with both title and summary blank/null remain in the final output.
- Primary model predicts 4-digit FoR Group codes.
- Only unresolved primary rows are sent to the fallback Division model.
- Fallback output is reduced to the 2-digit Division code.
- Rows unresolved by both models remain in the output with native nulls in `category_id`, `score_type`, and `score`.
- Produces **one consolidated Parquet output** for the whole backfill.
- Saves output locally only; no S3 write is performed in this notebook.


## 1. Imports, project paths and preprocessing module


In [ ]:
from pathlib import Path
from datetime import datetime
import logging
import re
import sys

import joblib
import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "models").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "models").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR.parent

LOCAL_BACKFILL_DIR = PROJECT_ROOT / "prod_data_temp"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "output"
DUPLICATE_DIR = DATA_DIR / "duplicates"
LOG_DIR = PROJECT_ROOT / "logs"

for d in [OUTPUT_DIR, DUPLICATE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_preparation import preprocess_text_fields

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOCAL_BACKFILL_DIR:", LOCAL_BACKFILL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Configuration

Normally only this cell should need reviewing.

The two model filenames below are taken from the previously used deployment notebook.


In [ ]:
MAIN_MODEL_PATH = PROJECT_ROOT / "models" / (
    "lemma_stop_neg_scale_10_mindf_5_maxdf_90_maxfold_5_min_pos_50_pos_prior_50_"
    "fields_of_research_negative_scaling_tfidf.joblib"
)

FALLBACK_MODEL_PATH = PROJECT_ROOT / "models" / (
    "lemma_stop_neg_scale_10_mindf_5_maxdf_90_maxfold_5_min_pos_50_pos_prior_50_"
    "fields_of_research_negative_scaling_tfidf_division.joblib"
)

ID_FIELDS = ["ApplicationID", "ApplicationOriginSource"]
MODEL_TEXT_FIELDS = ["ApplicationTitle", "ApplicationSummary"]

EXPECTED_BACKFILL_FILE_COUNT = 10
LOCAL_FILE_PATTERN = "*.parquet"

TAXONOMY_FILE_TOKEN = "FoR"
TAXONOMY_VALUE = "FieldsOfResearch"
MODEL_NAME = "FoRClassification"
MODEL_VERSION = "1.1"
SCORE_TYPE = "uncalibrated"

N_JOBS = 4
PREPROCESS_BATCH_SIZE = 250

# Local-only notebook: deliberately no S3 upload.
UPLOAD_OUTPUT_TO_S3 = False

print("MAIN_MODEL_PATH:", MAIN_MODEL_PATH)
print("FALLBACK_MODEL_PATH:", FALLBACK_MODEL_PATH)
print("UPLOAD_OUTPUT_TO_S3:", UPLOAD_OUTPUT_TO_S3)


## 3. Runtime compatibility and path checks

The joblib bundles should be run with the same scikit-learn/runtime family used by the working inference environment.
This cell reports versions and fails early if required local files are missing.

If your working inference environment uses Python 3.13.9 and scikit-learn 1.8.0, select that same kernel before running the notebook.


In [ ]:
import sklearn
import scipy
import pyarrow

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("joblib:", joblib.__version__)
print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)

if not LOCAL_BACKFILL_DIR.exists():
    raise FileNotFoundError(f"Backfill folder not found: {LOCAL_BACKFILL_DIR}")

for p in [MAIN_MODEL_PATH, FALLBACK_MODEL_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Model not found: {p}")

session_run_timestamp = datetime.now()
session_run_stamp = session_run_timestamp.strftime("%Y%m%d_%H%M%S")

log_path = LOG_DIR / f"for_local_backfill_{session_run_stamp}.log"
logger = logging.getLogger("for_local_backfill")
logger.setLevel(logging.INFO)
logger.handlers.clear()

fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

fh = logging.FileHandler(log_path)
fh.setFormatter(fmt)
logger.addHandler(fh)

sh = logging.StreamHandler()
sh.setFormatter(fmt)
logger.addHandler(sh)

logger.info("Started local backfill notebook")
print("Log:", log_path)


## 4. Discover the local historical Parquet files


In [ ]:
files_to_process = sorted(LOCAL_BACKFILL_DIR.glob(LOCAL_FILE_PATTERN))

print("Local Parquet files discovered:", len(files_to_process))
for f in files_to_process:
    print(" -", f.name)

if len(files_to_process) != EXPECTED_BACKFILL_FILE_COUNT:
    raise ValueError(
        f"Expected exactly {EXPECTED_BACKFILL_FILE_COUNT} local backfill files, "
        f"found {len(files_to_process)} in {LOCAL_BACKFILL_DIR}"
    )


## 5. Load and validate both single-joblib model bundles


In [ ]:
def load_bundle(path: Path) -> dict:
    bundle = joblib.load(path)

    if not isinstance(bundle, dict):
        raise TypeError(f"{path.name} must contain a dict")

    required = {"vectorizer", "models", "thresholds", "mlb"}
    missing = required - set(bundle)
    if missing:
        raise KeyError(f"{path.name} missing keys: {sorted(missing)}")

    thresholds = np.asarray(bundle["thresholds"]).reshape(-1)

    n_models = len(bundle["models"])
    n_thresholds = len(thresholds)
    n_classes = len(bundle["mlb"].classes_)

    if not (n_models == n_thresholds == n_classes):
        raise ValueError(
            f"Incompatible bundle lengths in {path.name}: "
            f"models={n_models}, thresholds={n_thresholds}, classes={n_classes}"
        )

    bundle = dict(bundle)
    bundle["thresholds"] = thresholds
    return bundle


primary_model = load_bundle(MAIN_MODEL_PATH)
fallback_model = load_bundle(FALLBACK_MODEL_PATH)

print("Primary classes:", len(primary_model["mlb"].classes_))
print("Fallback classes:", len(fallback_model["mlb"].classes_))
print("Example primary classes:", list(primary_model["mlb"].classes_[:5]))
print("Example fallback classes:", list(fallback_model["mlb"].classes_[:5]))


## 6. Reusable inference helpers

`positive_probability()` mirrors the working Version 2 approach: it calls each stored binary estimator's `predict_proba()` and takes the positive-class probability.


In [ ]:
def positive_probability(model, X):
    p = np.asarray(model.predict_proba(X))

    if p.ndim == 1:
        return p
    if p.shape[1] == 1:
        return p[:, 0]

    return p[:, 1]


def perform_inference(bundle: dict, df_in: pd.DataFrame):
    n_classes = len(bundle["mlb"].classes_)

    if len(df_in) == 0:
        return (
            np.empty((0, n_classes), dtype=float),
            np.empty((0, n_classes), dtype=np.int8),
        )

    X = bundle["vectorizer"].transform(df_in["PROCESSED_TEXT"])

    probs = np.column_stack(
        [positive_probability(model, X) for model in bundle["models"]]
    )

    preds = (
        probs >= bundle["thresholds"].reshape(1, -1)
    ).astype(np.int8)

    expected_shape = (len(df_in), n_classes)
    if probs.shape != expected_shape:
        raise ValueError(
            f"Unexpected probability shape {probs.shape}; expected {expected_shape}"
        )

    return probs, preds


def category_code(label, digits: int) -> str:
    match = re.match(rf"^\s*(\d{{{digits}}})\b", str(label))
    if not match:
        raise ValueError(
            f"Cannot extract {digits}-digit category code from label {label!r}"
        )
    return match.group(1)


def predictions_to_long(
    source_df: pd.DataFrame,
    probs: np.ndarray,
    preds: np.ndarray,
    bundle: dict,
    digits: int,
    source_name: str,
) -> pd.DataFrame:

    rows = []
    source_df = source_df.reset_index(drop=True)

    for row_pos, row in source_df.iterrows():
        selected = np.flatnonzero(preds[row_pos] == 1)

        for class_idx in selected:
            rows.append(
                {
                    "ApplicationID": row["ApplicationID"],
                    "ApplicationOriginSource": row["ApplicationOriginSource"],
                    "category_id": category_code(
                        bundle["mlb"].classes_[class_idx],
                        digits,
                    ),
                    "score": float(probs[row_pos, class_idx]),
                    "prediction_source": source_name,
                }
            )

    return pd.DataFrame(
        rows,
        columns=ID_FIELDS + ["category_id", "score", "prediction_source"],
    )


def blank(series: pd.Series) -> pd.Series:
    return (
        series.isna()
        | series.astype("string").fillna("").str.strip().eq("")
    )


## 7. Process one local file

Important behaviour:

- Duplicate application identities already seen in an earlier backfill file are excluded.
- Both-null/blank title+summary rows are excluded **only from inference** and still flow to final output.
- Fallback model receives only records unresolved by the primary model.
- No file is uploaded to S3.


In [ ]:
def process_local_file(input_path: Path, seen_keys: set):
    started = datetime.now()
    logger.info("Processing %s", input_path.name)

    # A. Read local Parquet
    df_raw = pd.read_parquet(input_path, engine="pyarrow")

    required = ID_FIELDS + MODEL_TEXT_FIELDS
    missing = [c for c in required if c not in df_raw.columns]
    if missing:
        raise ValueError(f"{input_path.name} missing required columns: {missing}")

    # B. Drop duplicates within this file
    within_dup_mask = df_raw.duplicated(subset=ID_FIELDS, keep="first")
    df_within_duplicates = df_raw.loc[within_dup_mask].copy()
    df_unique = (
        df_raw.drop_duplicates(subset=ID_FIELDS, keep="first")
        .reset_index(drop=True)
    )

    # C. Drop identities already processed from earlier backfill files
    current_keys = list(
        zip(
            df_unique["ApplicationID"].astype("string"),
            df_unique["ApplicationOriginSource"].astype("string"),
        )
    )
    already_seen_mask = pd.Series(
        [key in seen_keys for key in current_keys],
        index=df_unique.index,
    )

    df_cross_file_duplicates = df_unique.loc[already_seen_mask].copy()
    df_base = df_unique.loc[~already_seen_mask].copy().reset_index(drop=True)

    # Audit duplicate records locally
    df_duplicates = pd.concat(
        [df_within_duplicates, df_cross_file_duplicates],
        ignore_index=True,
    )
    if len(df_duplicates):
        duplicate_path = DUPLICATE_DIR / f"{input_path.stem}_duplicates.parquet"
        df_duplicates.to_parquet(
            duplicate_path,
            index=False,
            engine="pyarrow",
        )

    # Mark retained application identities as seen
    retained_keys = list(
        zip(
            df_base["ApplicationID"].astype("string"),
            df_base["ApplicationOriginSource"].astype("string"),
        )
    )
    seen_keys.update(retained_keys)

    # Complete unique population for this file
    df_output_base = df_base[ID_FIELDS].copy()

    # D. Exclude no-text rows only from inference
    no_text_mask = (
        blank(df_base["ApplicationTitle"])
        & blank(df_base["ApplicationSummary"])
    )

    df_model_input = (
        df_base.loc[~no_text_mask]
        .copy()
        .reset_index(drop=True)
    )

    # E. Shared preprocessing
    if len(df_model_input):
        df_cleaned = preprocess_text_fields(
            df=df_model_input.copy(),
            text_fields=MODEL_TEXT_FIELDS,
            new_field_name="PROCESSED_TEXT",
            n_jobs=N_JOBS,
            batch_size=PREPROCESS_BATCH_SIZE,
        )

        if "PROCESSED_TEXT" not in df_cleaned.columns:
            raise KeyError("PROCESSED_TEXT was not created by preprocessing")
    else:
        df_cleaned = df_model_input.copy()
        df_cleaned["PROCESSED_TEXT"] = pd.Series(dtype="string")

    # F. Primary 4-digit Group model
    p_probs, p_preds = perform_inference(primary_model, df_cleaned)
    p_counts = (
        p_preds.sum(axis=1)
        if len(df_cleaned)
        else np.array([], dtype=int)
    )
    p_resolved = p_counts > 0

    df_primary = predictions_to_long(
        df_cleaned,
        p_probs,
        p_preds,
        primary_model,
        digits=4,
        source_name="PRIMARY_GROUP",
    )

    # G. Fallback 2-digit Division model on primary-unresolved only
    df_unresolved = (
        df_cleaned.loc[~p_resolved]
        .copy()
        .reset_index(drop=True)
    )

    f_probs, f_preds = perform_inference(fallback_model, df_unresolved)
    f_counts = (
        f_preds.sum(axis=1)
        if len(df_unresolved)
        else np.array([], dtype=int)
    )

    df_fallback = predictions_to_long(
        df_unresolved,
        f_probs,
        f_preds,
        fallback_model,
        digits=2,
        source_name="FALLBACK_DIVISION",
    )

    # H. Merge model predictions onto COMPLETE retained population
    df_predictions = pd.concat(
        [df_primary, df_fallback],
        ignore_index=True,
    )

    df_output = df_output_base.merge(
        df_predictions,
        on=ID_FIELDS,
        how="left",
        validate="one_to_many",
    )

    # I. Final stakeholder schema
    model_run_datetime = pd.Timestamp.now()

    df_output["model_run_date"] = model_run_datetime

    prediction_exists = df_output["category_id"].notna()

    df_output["score_type"] = pd.Series(
        pd.NA,
        index=df_output.index,
        dtype="string",
    )
    df_output.loc[prediction_exists, "score_type"] = SCORE_TYPE

    # category_id is varchar-compatible / nullable string
    df_output["category_id"] = df_output["category_id"].astype("string")

    # Nullable decimal-compatible score
    df_output["score"] = (
        pd.to_numeric(df_output["score"], errors="coerce")
        .round(2)
        .astype("Float64")
    )

    df_output["Taxonomy"] = TAXONOMY_VALUE
    df_output["model_name"] = MODEL_NAME
    df_output["model_version"] = MODEL_VERSION

    # For any unresolved/no-text prediction:
    # category_id, score_type and score must all be native null.
    null_pred = df_output["category_id"].isna()
    df_output.loc[null_pred, "category_id"] = pd.NA
    df_output.loc[null_pred, "score_type"] = pd.NA
    df_output.loc[null_pred, "score"] = pd.NA

    FINAL_COLUMNS = [
        "ApplicationID",
        "ApplicationOriginSource",
        "model_run_date",
        "category_id",
        "score_type",
        "score",
        "Taxonomy",
        "model_name",
        "model_version",
    ]

    df_output = df_output[FINAL_COLUMNS]

    string_columns = [
        "ApplicationID",
        "ApplicationOriginSource",
        "category_id",
        "score_type",
        "Taxonomy",
        "model_name",
        "model_version",
    ]
    for c in string_columns:
        df_output[c] = df_output[c].astype("string")

    # J. File-level quality checks
    unique_input = len(df_output_base)
    unique_output = df_output[ID_FIELDS].drop_duplicates().shape[0]

    if unique_input != unique_output:
        raise ValueError(
            f"Completeness failure in {input_path.name}: "
            f"input={unique_input}, output={unique_output}"
        )

    predicted = df_output["category_id"].notna()
    if (
        df_output.loc[predicted, "score_type"].isna().any()
        or df_output.loc[predicted, "score"].isna().any()
    ):
        raise ValueError("Predicted rows contain null score_type/score")

    unresolved = df_output["category_id"].isna()
    if df_output.loc[
        unresolved, ["category_id", "score_type", "score"]
    ].notna().any().any():
        raise ValueError(
            "Unresolved rows must have category_id, score_type and score all null"
        )

    result = {
        "input_file": input_path.name,
        "raw_input_rows": int(len(df_raw)),
        "duplicates_removed": int(len(df_duplicates)),
        "unique_applications_retained": int(unique_input),
        "both_text_fields_missing": int(no_text_mask.sum()),
        "primary_resolved_applications": int(p_resolved.sum()),
        "sent_to_fallback": int(len(df_unresolved)),
        "fallback_resolved_applications": (
            int((f_counts > 0).sum()) if len(f_counts) else 0
        ),
        "final_null_prediction_applications": int(
            df_output.loc[
                unresolved, ID_FIELDS
            ].drop_duplicates().shape[0]
        ),
        "final_output_rows": int(len(df_output)),
        "started_at": started.isoformat(timespec="seconds"),
        "finished_at": datetime.now().isoformat(timespec="seconds"),
    }

    logger.info("Completed %s | %s", input_path.name, result)
    return result, df_output


## 8. Execute all 10 local historical files

Files are processed sequentially.  
The output DataFrames are retained and combined only after all files complete successfully.


In [ ]:
run_results = []
all_output_dfs = []
seen_application_keys = set()

for i, input_path in enumerate(files_to_process, start=1):
    print(f"\n[{i}/{len(files_to_process)}] {input_path.name}")

    try:
        result, output_df = process_local_file(
            input_path=input_path,
            seen_keys=seen_application_keys,
        )

        run_results.append(result)
        all_output_dfs.append(output_df)

        print(
            "Completed:",
            input_path.name,
            "| retained applications:",
            result["unique_applications_retained"],
            "| output rows:",
            result["final_output_rows"],
        )

    except Exception:
        logger.exception("FAILED %s", input_path.name)
        raise

print("\nAll local backfill files processed successfully.")


## 9. Combine all files into one backfill output


In [ ]:
if not all_output_dfs:
    raise RuntimeError("No output DataFrames were produced")

df_backfill_output = pd.concat(
    all_output_dfs,
    ignore_index=True,
)

# Defensive duplicate checks/removal.
# Predicted rows: one row per ApplicationID + OriginSource + category_id.
predicted_mask = df_backfill_output["category_id"].notna()

df_predicted = (
    df_backfill_output.loc[predicted_mask]
    .drop_duplicates(
        subset=ID_FIELDS + ["category_id"],
        keep="first",
    )
)

# Unresolved rows: exactly one row per application identity.
df_unresolved = (
    df_backfill_output.loc[~predicted_mask]
    .drop_duplicates(
        subset=ID_FIELDS,
        keep="first",
    )
)

df_backfill_output = pd.concat(
    [df_predicted, df_unresolved],
    ignore_index=True,
)

print("Combined final rows:", len(df_backfill_output))
print(
    "Combined unique applications:",
    df_backfill_output[ID_FIELDS].drop_duplicates().shape[0],
)
print(
    "Native-null applications:",
    df_backfill_output.loc[
        df_backfill_output["category_id"].isna(),
        ID_FIELDS,
    ].drop_duplicates().shape[0],
)


## 10. Final completeness and schema validation


In [ ]:
# The number of unique output applications must equal the number retained
# after global duplicate removal across the ten source files.
expected_unique_applications = len(seen_application_keys)
actual_unique_applications = (
    df_backfill_output[ID_FIELDS]
    .drop_duplicates()
    .shape[0]
)

print("Expected unique applications:", expected_unique_applications)
print("Actual unique applications:", actual_unique_applications)

if expected_unique_applications != actual_unique_applications:
    raise ValueError(
        "Final completeness failure: "
        f"expected {expected_unique_applications}, "
        f"found {actual_unique_applications}"
    )

# Prediction-result null rule
unresolved = df_backfill_output["category_id"].isna()

if df_backfill_output.loc[
    unresolved,
    ["category_id", "score_type", "score"],
].notna().any().any():
    raise ValueError(
        "Unresolved applications must have native null "
        "category_id, score_type and score"
    )

# Predicted row uniqueness
if df_backfill_output.loc[
    ~unresolved
].duplicated(
    subset=ID_FIELDS + ["category_id"]
).any():
    raise ValueError(
        "Duplicate application/source/category prediction rows remain"
    )

# Unresolved row uniqueness
if df_backfill_output.loc[
    unresolved
].duplicated(
    subset=ID_FIELDS
).any():
    raise ValueError(
        "Duplicate unresolved application rows remain"
    )

print("\nFinal dtypes:")
print(df_backfill_output.dtypes)

display(df_backfill_output.head(10))
display(df_backfill_output.loc[unresolved].head(10))


## 11. Save ONE consolidated Parquet file locally

This notebook intentionally performs no S3 write.

Filename convention used here: `DateTime_yyyyMMddHHmm_FoR.parquet`.


In [ ]:
output_timestamp = datetime.now().strftime("%Y%m%d%H%M")
output_filename = f"{output_timestamp}_{TAXONOMY_FILE_TOKEN}.parquet"
local_output_path = OUTPUT_DIR / output_filename

df_backfill_output.to_parquet(
    local_output_path,
    index=False,
    engine="pyarrow",
)

if not local_output_path.exists():
    raise FileNotFoundError(
        f"Local output was not created: {local_output_path}"
    )

# Round-trip validation
df_roundtrip = pd.read_parquet(
    local_output_path,
    engine="pyarrow",
)

if len(df_roundtrip) != len(df_backfill_output):
    raise ValueError(
        "Parquet round-trip row-count mismatch: "
        f"before={len(df_backfill_output)}, "
        f"after={len(df_roundtrip)}"
    )

print("Saved consolidated local backfill output:")
print(local_output_path)
print("Rows:", len(df_roundtrip))
print(
    "Unique applications:",
    df_roundtrip[ID_FIELDS].drop_duplicates().shape[0],
)
print("File size bytes:", local_output_path.stat().st_size)


## 12. Run summary


In [ ]:
df_run_summary = pd.DataFrame(run_results)

display(df_run_summary)

print("Files processed:", len(df_run_summary))
print("Total raw rows:", int(df_run_summary["raw_input_rows"].sum()))
print("Total duplicates removed:", int(df_run_summary["duplicates_removed"].sum()))
print(
    "Total retained unique applications:",
    int(df_run_summary["unique_applications_retained"].sum()),
)
print(
    "Total no-text applications:",
    int(df_run_summary["both_text_fields_missing"].sum()),
)
print(
    "Total sent to fallback:",
    int(df_run_summary["sent_to_fallback"].sum()),
)
print(
    "Total final native-null applications:",
    int(df_run_summary["final_null_prediction_applications"].sum()),
)


## After this one-off backfill

This notebook as the historical backfill runner only.

For weekly production DELTA processing, continue using the separate S3-based notebook so the normal timestamp-based "latest unprocessed file" logic remains isolated from this one-off legacy-file workflow.
